# 馃彔 Predicci贸n de Precios de Casas
## Regresi贸n con Machine Learning

---

En este ejercicio vamos a construir un modelo de **Machine Learning** que predice el precio de una casa seg煤n sus caracter铆sticas.

A diferencia de los ejercicios de **clasificaci贸n** (donde la salida es una categor铆a, como "setosa" o "virginica"), aqu铆 la salida es un **n煤mero continuo**: el precio en d贸lares. Eso se llama **regresi贸n**.

### 馃彙 驴Qu茅 informaci贸n vamos a usar?
- Metros cuadrados
- Cantidad de habitaciones
- Cantidad de ba帽os
- Antig眉edad de la casa
- Distancia al centro (km)
- Si tiene garage o no
- Si tiene pileta o no

### 馃幆 Objetivos:
- Entender la diferencia entre clasificaci贸n y regresi贸n
- Generar y explorar un dataset realista
- Entrenar modelos de regresi贸n
- Interpretar m茅tricas como RMSE y R虏
- Predecir el precio de una casa nueva

### 馃椇锔?Estructura:
1. Importar librer铆as
2. Crear el dataset
3. Explorar y visualizar los datos
4. Preparar los datos
5. Entrenar modelos
6. Evaluar y comparar
7. Hacer predicciones
8. 馃弳 Desaf铆os extra

---
## 馃摝 Paso 1: Importar librer铆as

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Modelos de regresi贸n
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# M茅tricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

print('鉁?Librer铆as importadas correctamente')

---
## 馃彈锔?Paso 2: Crear el dataset

Vamos a generar un dataset sint茅tico pero **realista**, con relaciones l贸gicas entre las variables:
m谩s metros cuadrados 鈫?precio m谩s alto, m谩s distancia al centro 鈫?precio m谩s bajo, etc.

In [ ]:
N = 500  # cantidad de casas

# Caracter铆sticas de las casas
metros     = np.random.randint(40, 300, N).astype(float)
hab        = np.random.randint(1, 6, N).astype(float)
banios     = np.random.randint(1, 4, N).astype(float)
antiguedad = np.random.randint(0, 50, N).astype(float)
distancia  = np.round(np.random.uniform(0.5, 30, N), 1)
garage     = np.random.choice([0, 1], N, p=[0.4, 0.6]).astype(float)
pileta     = np.random.choice([0, 1], N, p=[0.7, 0.3]).astype(float)

# Precio: combinaci贸n realista de caracter铆sticas + algo de ruido
precio = (
      1200  * metros
    + 8000  * hab
    + 6000  * banios
    - 500   * antiguedad
    - 1500  * distancia
    + 15000 * garage
    + 20000 * pileta
    + np.random.normal(0, 15000, N)   # ruido aleatorio
    + 50000                           # precio base
)

# Aseguramos que ning煤n precio sea negativo
precio = np.maximum(precio, 30000)

# Crear DataFrame
df = pd.DataFrame({
    'metros_cuadrados': metros,
    'habitaciones':     hab,
    'banios':           banios,
    'antiguedad_a帽os':  antiguedad,
    'distancia_centro': distancia,
    'garage':           garage,
    'pileta':           pileta,
    'precio_usd':       precio.round(-2)   # redondeamos a centenas
})

print('馃搳 Primeras filas del dataset:')
print(df.head(10).to_string(index=False))
print(f'\nTotal de casas: {len(df)}')

---
## 馃搳 Paso 3: Explorar y visualizar los datos

In [ ]:
# Estad铆sticas descriptivas
print('馃搱 Estad铆sticas del dataset:')
df.describe().round(1)

In [ ]:
# Distribuci贸n del precio
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Distribuci贸n del precio de las casas', fontsize=13, fontweight='bold')

axes[0].hist(df['precio_usd'] / 1000, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Precio (miles de USD)')
axes[0].set_ylabel('Cantidad de casas')
axes[0].set_title('Histograma de precios')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df['precio_usd'] / 1000, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_ylabel('Precio (miles de USD)')
axes[1].set_title('Boxplot de precios')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'   Precio m铆nimo:  USD {df["precio_usd"].min():,.0f}')
print(f'   Precio m谩ximo:  USD {df["precio_usd"].max():,.0f}')
print(f'   Precio promedio: USD {df["precio_usd"].mean():,.0f}')

In [ ]:
# Relaci贸n entre caracter铆sticas num茅ricas y precio
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Relaci贸n entre caracter铆sticas y precio', fontsize=13, fontweight='bold')

pares = [
    ('metros_cuadrados', 'Metros cuadrados'),
    ('habitaciones',     'Habitaciones'),
    ('antiguedad_a帽os',  'Antig眉edad (a帽os)'),
    ('distancia_centro', 'Distancia al centro (km)'),
]

for ax, (col, titulo) in zip(axes.flat, pares):
    ax.scatter(df[col], df['precio_usd'] / 1000,
               alpha=0.4, color='steelblue', edgecolors='none', s=30)
    # L铆nea de tendencia
    z = np.polyfit(df[col], df['precio_usd'] / 1000, 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Tendencia')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Precio (miles USD)')
    ax.set_title(f'{titulo} vs Precio')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('馃挕 La l铆nea roja muestra la tendencia lineal de cada relaci贸n.')

In [ ]:
# Efecto de garage y pileta
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Impacto de amenidades en el precio', fontsize=13, fontweight='bold')

for ax, (col, titulo, etiquetas) in zip(axes, [
    ('garage', 'Garage', ['Sin garage', 'Con garage']),
    ('pileta', 'Pileta', ['Sin pileta', 'Con pileta']),
]):
    grupos = [df[df[col] == 0]['precio_usd'] / 1000,
              df[df[col] == 1]['precio_usd'] / 1000]
    bp = ax.boxplot(grupos, labels=etiquetas, patch_artist=True)
    bp['boxes'][0].set_facecolor('#AED6F1')
    bp['boxes'][1].set_facecolor('#2ECC71')
    ax.set_ylabel('Precio (miles USD)')
    ax.set_title(f'Precio seg煤n {titulo}')
    ax.grid(True, axis='y', alpha=0.3)

    # Mostrar promedio
    for i, g in enumerate(grupos):
        ax.text(i + 1, g.max() * 0.98, f'Prom: {g.mean():.0f}k',
                ha='center', fontsize=9, color='darkblue')

plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlaci贸n
fig, ax = plt.subplots(figsize=(8, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, square=True)
ax.set_title('Correlaci贸n entre variables', fontweight='bold')
plt.tight_layout()
plt.show()

print('馃挕 La 煤ltima fila muestra qu茅 tan relacionada est谩 cada variable con el precio.')
corr_precio = corr['precio_usd'].drop('precio_usd').sort_values(ascending=False)
print('\nCorrelaci贸n con precio_usd:')
for var, val in corr_precio.items():
    barra = '鈻? * int(abs(val) * 20)
    signo = '+' if val >= 0 else '-'
    print(f'   {var:<22} {signo}{barra} {val:.3f}')

---
## 鈿欙笍 Paso 4: Preparar los datos

In [ ]:
# Separar features y target
FEATURES = ['metros_cuadrados', 'habitaciones', 'banios',
            'antiguedad_a帽os', 'distancia_centro', 'garage', 'pileta']

X = df[FEATURES].values
y = df['precio_usd'].values

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalar los datos
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('馃搳 Divisi贸n del dataset:')
print(f'   Entrenamiento: {X_train.shape[0]} casas')
print(f'   Prueba:        {X_test.shape[0]} casas')
print(f'   Caracter铆sticas: {len(FEATURES)}')
print('\n鉁?Datos listos para entrenar')

---
## 馃 Paso 5: Entrenar modelos

Vamos a entrenar **4 modelos de regresi贸n** y comparar cu谩l predice mejor los precios.

### 馃搻 M茅tricas que vamos a usar:

| M茅trica | Significado |
|---|---|
| **MAE** | Error absoluto promedio (en USD). Cu谩nto se equivoca el modelo en promedio. |
| **RMSE** | Ra铆z del error cuadr谩tico medio. Penaliza m谩s los errores grandes. |
| **R虏** | Coeficiente de determinaci贸n. Va de 0 a 1 鈫?cu谩nto explica el modelo. |

In [ ]:
modelos = {
    'Regresi贸n Lineal':     LinearRegression(),
    'Ridge':                Ridge(alpha=1.0),
    '脕rbol de Decisi贸n':    DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados = {}

print(f'{"Modelo":<25}  {"MAE":>10}  {"RMSE":>10}  {"R虏":>8}')
print('-' * 58)

for nombre, modelo in modelos.items():
    modelo.fit(X_train_sc, y_train)
    y_pred = modelo.predict(X_test_sc)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    resultados[nombre] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'y_pred': y_pred}
    print(f'{nombre:<25}  ${mae:>9,.0f}  ${rmse:>9,.0f}  {r2:>8.4f}')

mejor = max(resultados, key=lambda k: resultados[k]['R2'])
print(f'\n馃弳 Mejor modelo (mayor R虏): {mejor}')

---
## 馃搳 Paso 6: Evaluar y comparar modelos

In [ ]:
# Comparaci贸n visual de R虏 y MAE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparaci贸n de modelos', fontsize=13, fontweight='bold')

nombres  = list(resultados.keys())
r2s      = [resultados[n]['R2']  for n in nombres]
maes     = [resultados[n]['MAE'] / 1000 for n in nombres]
cols     = ['#3498DB', '#9B59B6', '#E74C3C', '#2ECC71', '#F39C12']

# R虏
bars = axes[0].bar(nombres, r2s, color=cols, edgecolor='white')
axes[0].set_title('R虏 (m谩s alto = mejor)', fontweight='bold')
axes[0].set_ylabel('R虏')
axes[0].set_ylim([0, 1.1])
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, r2s):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)

# MAE
bars = axes[1].bar(nombres, maes, color=cols, edgecolor='white')
axes[1].set_title('MAE (m谩s bajo = mejor)', fontweight='bold')
axes[1].set_ylabel('Error promedio (miles USD)')
axes[1].grid(True, axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, maes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'${val:.1f}k', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Gr谩fico: Precio real vs Precio predicho (mejor modelo)
y_pred_mejor = resultados[mejor]['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'An谩lisis del mejor modelo: {mejor}', fontsize=13, fontweight='bold')

# Real vs Predicho
axes[0].scatter(y_test / 1000, y_pred_mejor / 1000,
                alpha=0.5, color='steelblue', edgecolors='none', s=40)
lim = [min(y_test.min(), y_pred_mejor.min()) / 1000,
       max(y_test.max(), y_pred_mejor.max()) / 1000]
axes[0].plot(lim, lim, 'r--', linewidth=2, label='Predicci贸n perfecta')
axes[0].set_xlabel('Precio real (miles USD)')
axes[0].set_ylabel('Precio predicho (miles USD)')
axes[0].set_title('Real vs Predicho')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribuci贸n de errores
errores = (y_pred_mejor - y_test) / 1000
axes[1].hist(errores, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Error = 0')
axes[1].set_xlabel('Error de predicci贸n (miles USD)')
axes[1].set_ylabel('Cantidad de casas')
axes[1].set_title('Distribuci贸n de errores')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('馃挕 En el gr谩fico izquierdo, cuanto m谩s cerca est茅n los puntos de la l铆nea roja,\n'
      '   mejor es la predicci贸n del modelo.')

In [ ]:
# Coeficientes de la Regresi贸n Lineal (muy interpretables)
lr = modelos['Regresi贸n Lineal']
coefs = pd.Series(lr.coef_, index=FEATURES).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colores_coef = ['#2ECC71' if c > 0 else '#E74C3C' for c in coefs]
bars = ax.bar(coefs.index, coefs.values, color=colores_coef, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes de la Regresi贸n Lineal\n(escala normalizada)', fontweight='bold')
ax.set_ylabel('Coeficiente')
ax.tick_params(axis='x', rotation=20)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('馃挕 Verde = aumenta el precio  |  Rojo = disminuye el precio')
print('   (Los valores est谩n normalizados, no representan USD directamente)')

---
## 馃敭 Paso 7: Hacer predicciones con casas nuevas

In [ ]:
def predecir_precio(metros, habitaciones, banios, antiguedad,
                    distancia, garage, pileta, modelo_nombre=None):
    """
    Predice el precio de una casa.

    Par谩metros:
        metros        : metros cuadrados
        habitaciones  : cantidad de habitaciones
        banios        : cantidad de ba帽os
        antiguedad    : a帽os de antig眉edad
        distancia     : distancia al centro en km
        garage        : 1 si tiene, 0 si no
        pileta        : 1 si tiene, 0 si no
        modelo_nombre : nombre del modelo a usar
    """
    if modelo_nombre is None:
        modelo_nombre = mejor

    casa = np.array([[metros, habitaciones, banios, antiguedad,
                      distancia, garage, pileta]])
    casa_sc = scaler.transform(casa)
    precio_pred = modelos[modelo_nombre].predict(casa_sc)[0]

    print(f'馃彙 Caracter铆sticas de la casa:')
    print(f'   Metros cuadrados : {metros} m虏')
    print(f'   Habitaciones     : {habitaciones}')
    print(f'   Ba帽os            : {banios}')
    print(f'   Antig眉edad       : {antiguedad} a帽os')
    print(f'   Distancia centro : {distancia} km')
    print(f'   Garage           : {"S铆" if garage else "No"}')
    print(f'   Pileta           : {"S铆" if pileta else "No"}')
    print(f'\n馃 Modelo: {modelo_nombre}')
    print(f'馃挵 Precio estimado: USD {precio_pred:,.0f}')
    print()


print('=== PREDICCIONES DE EJEMPLO ===\n')

# Casa peque帽a, lejos del centro
predecir_precio(
    metros=60, habitaciones=2, banios=1,
    antiguedad=20, distancia=15, garage=0, pileta=0
)

# Casa mediana
predecir_precio(
    metros=120, habitaciones=3, banios=2,
    antiguedad=10, distancia=8, garage=1, pileta=0
)

# Casa grande y moderna
predecir_precio(
    metros=250, habitaciones=5, banios=3,
    antiguedad=2, distancia=2, garage=1, pileta=1
)

In [ ]:
# 馃敡 隆Prob谩 con tu propia casa!

predecir_precio(
    metros        = 100,   # 鈫?modific谩
    habitaciones  = 3,     # 鈫?modific谩
    banios        = 2,     # 鈫?modific谩
    antiguedad    = 5,     # 鈫?modific谩 (a帽os)
    distancia     = 5.0,   # 鈫?modific谩 (km)
    garage        = 1,     # 鈫?1 = s铆, 0 = no
    pileta        = 0      # 鈫?1 = s铆, 0 = no
)

---
## 馃弳 Desaf铆os Extra

---
### 馃 Desaf铆o 1 (F谩cil): 驴Qu茅 pasa si agreg谩s una caracter铆stica?

Agreg谩 una variable nueva al dataset, como la cantidad de pisos o si est谩 en barrio cerrado, y volv茅 a entrenar. 驴Mejora el R虏?

In [ ]:
# DESAF脥O 1: Agregar una nueva caracter铆stica

df2 = df.copy()

# Nueva variable: 驴est谩 en barrio cerrado?
df2['barrio_cerrado'] = np.random.choice([0, 1], N, p=[0.6, 0.4]).astype(float)

# Ajustamos el precio para que tenga efecto real
df2.loc[df2['barrio_cerrado'] == 1, 'precio_usd'] += np.random.normal(25000, 5000,
    df2['barrio_cerrado'].sum())

FEATURES_2 = FEATURES + ['barrio_cerrado']
X2 = df2[FEATURES_2].values
y2 = df2['precio_usd'].values

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
X2_tr_sc = scaler2.fit_transform(X2_tr)
X2_te_sc = scaler2.transform(X2_te)

lr2 = LinearRegression()
lr2.fit(X2_tr_sc, y2_tr)
r2_nuevo = r2_score(y2_te, lr2.predict(X2_te_sc))
r2_orig  = resultados['Regresi贸n Lineal']['R2']

print(f'Regresi贸n Lineal 鈥?R虏 original  : {r2_orig:.4f}')
print(f'Regresi贸n Lineal 鈥?R虏 con nueva : {r2_nuevo:.4f}')
print(f'Cambio: {(r2_nuevo - r2_orig):+.4f}')

---
### 馃 Desaf铆o 2 (Medio): Comparar todos los modelos gr谩ficamente

In [ ]:
# DESAF脥O 2: Visualizar real vs predicho para todos los modelos

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Real vs Predicho 鈥?Todos los modelos', fontsize=14, fontweight='bold')

for ax, (nombre, res) in zip(axes.flat, resultados.items()):
    ax.scatter(y_test / 1000, res['y_pred'] / 1000,
               alpha=0.5, s=25, color='steelblue', edgecolors='none')
    lim = [min(y_test.min(), res['y_pred'].min()) / 1000,
           max(y_test.max(), res['y_pred'].max()) / 1000]
    ax.plot(lim, lim, 'r--', linewidth=1.5)
    ax.set_title(f'{nombre}\nR虏={res["R2"]:.3f}  MAE=${res["MAE"]/1000:.1f}k',
                 fontsize=10)
    ax.set_xlabel('Real (miles USD)')
    ax.set_ylabel('Predicho (miles USD)')
    ax.grid(True, alpha=0.3)

# Ocultar el 煤ltimo subplot vac铆o
axes.flat[-1].set_visible(False)

plt.tight_layout()
plt.show()

---
### 馃 Desaf铆o 3 (Dif铆cil): An谩lisis de sensibilidad

驴Cu谩nto cambia el precio si modific谩s una sola variable, manteniendo las dem谩s fijas?

In [ ]:
# DESAF脥O 3: An谩lisis de sensibilidad 鈥?驴c贸mo var铆a el precio?

# Casa base
base = {'metros_cuadrados': 100, 'habitaciones': 3, 'banios': 2,
        'antiguedad_a帽os': 10, 'distancia_centro': 5, 'garage': 1, 'pileta': 0}

modelo_sens = modelos[mejor]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'Sensibilidad del precio seg煤n cada variable\n(modelo: {mejor})',
             fontsize=13, fontweight='bold')

rangos = {
    'metros_cuadrados': np.arange(40, 300, 10),
    'habitaciones':     np.arange(1, 6, 1),
    'banios':           np.arange(1, 4, 1),
    'antiguedad_a帽os':  np.arange(0, 50, 2),
    'distancia_centro': np.arange(0.5, 30, 1),
}

for ax, (var, valores) in zip(axes.flat, rangos.items()):
    precios = []
    for v in valores:
        casa = base.copy()
        casa[var] = v
        arr = np.array([[casa[f] for f in FEATURES]])
        arr_sc = scaler.transform(arr)
        precios.append(modelo_sens.predict(arr_sc)[0] / 1000)

    ax.plot(valores, precios, color='steelblue', linewidth=2.5, marker='o', markersize=4)
    ax.axvline(base[var], color='red', linestyle='--', label=f'Base: {base[var]}')
    ax.set_title(var.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel(var)
    ax.set_ylabel('Precio estimado (miles USD)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

print('馃挕 La l铆nea roja marca el valor base de cada variable.')
print('   Pod茅s ver cu谩nto sube o baja el precio al modificar cada caracter铆stica.')

---
## 馃摑 Resumen y Conclusiones

### Lo que aprendiste:

| Concepto | Descripci贸n |
|---|---|
| **Regresi贸n** | Predecir un valor num茅rico continuo (vs clasificaci贸n que predice categor铆as) |
| **MAE** | Error absoluto promedio en la misma unidad que el target (USD) |
| **RMSE** | Penaliza m谩s los errores grandes que MAE |
| **R虏** | Qu茅 tan bien explica el modelo la variaci贸n del precio (0 a 1) |
| **Regresi贸n Lineal** | Modelo simple e interpretable, buena l铆nea de base |
| **Random Forest / Gradient Boosting** | Modelos m谩s poderosos para relaciones no lineales |
| **An谩lisis de sensibilidad** | C贸mo var铆a la predicci贸n al cambiar una variable |

### Para seguir aprendiendo:
- 馃摎 Prob谩 con el dataset real de Scikit-learn: `from sklearn.datasets import fetch_california_housing`
- 馃攳 Investig谩 **regularizaci贸n**: Lasso y Ridge controlan el sobreajuste en regresi贸n lineal
- 馃帗 Explor谩 la optimizaci贸n de hiperpar谩metros con `GridSearchCV`